# 03 — Sentiment Scoring
Scores each article with VADER (fast) or RoBERTa (accurate).
Input: `data/classified_articles.csv` → Output: `data/scored_articles.csv`

In [10]:
import pandas as pd

df = pd.read_csv('data/classified_articles.csv', parse_dates=['date'])
print(df.shape)

(12447, 8)


## Option A — VADER (recommended for this project)
Fast, no GPU, designed for news/social media. Outputs compound score in [-1, 1].

In [11]:
# pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def vader_score(text: str) -> float:
    # Use headline + first 300 chars of body — VADER degrades on long text
    snippet = str(text)[:300]
    return sia.polarity_scores(snippet)['compound']

df['text_sample'] = df['headline'] + '. ' + df['body'].str[:300]
df['sentiment'] = df['text_sample'].apply(vader_score)

# Map compound to label
df['sentiment_label'] = pd.cut(
    df['sentiment'],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=['negative', 'neutral', 'positive']
)

print(df['sentiment_label'].value_counts())
df[['headline', 'sentiment', 'sentiment_label']].head(5)

sentiment_label
positive    6504
negative    4971
neutral      972
Name: count, dtype: int64


,headline,sentiment,sentiment_label
0,People in the UK: have you received good or ba...,0.4404,positive
1,People in the UK: have you received good or ba...,0.4404,positive
2,People in the UK: have you received good or ba...,0.4404,positive
3,"Artificial intelligence will cost jobs, admits...",0.5994,positive
4,"Artificial intelligence will cost jobs, admits...",0.5994,positive


## Option B — RoBERTa (better accuracy, ~30 min on CPU for 10k)
Use `cardiffnlp/twitter-roberta-base-sentiment-latest` — trained on social/news text.

In [7]:
# from transformers import pipeline
# import torch
#
# device = 0 if torch.cuda.is_available() else -1
# sentiment_pipe = pipeline(
#     'sentiment-analysis',
#     model='cardiffnlp/twitter-roberta-base-sentiment-latest',
#     device=device,
#     truncation=True,
#     max_length=512
# )
#
# # Batch inference for speed
# texts = (df['headline'] + '. ' + df['body'].str[:300]).tolist()
# results = sentiment_pipe(texts, batch_size=32)
#
# label_map = {'positive': 1, 'neutral': 0, 'negative': -1}
# df['sentiment_label'] = [r['label'].lower() for r in results]
# df['sentiment'] = [label_map[r['label'].lower()] * r['score'] for r in results]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [12]:
df.to_csv('data/scored_articles.csv', index=False)
print('Saved: data/scored_articles.csv')
df.describe()

Saved: data/scored_articles.csv


,date,wordcount,sentiment
count,12447,12447.000000,12447.000000
mean,2024-03-31 09:12:11.453362,1124.469350,0.072285
min,2021-01-01 00:00:00,106.000000,-0.984600
25%,2023-06-03 00:00:00,538.000000,-0.452600
50%,2024-05-29 00:00:00,720.000000,0.102700
75%,2025-05-22 00:00:00,1049.500000,0.599400
max,2026-02-01 00:00:00,20815.000000,0.986100
std,NaN,1543.614446,0.579678
